# LSTM 验证集错误与指标分析

比较每轮验证损失和 F1，并检查按验证损失选择的 LSTM 模型在正负类别上的错误。测试集不参与分析。

In [ ]:
# 比较验证损失和 F1 对应的轮次
import pandas as pd

from sentiment_analysis.paths import TABLES_DIR

training_history = pd.read_csv(TABLES_DIR / "lstm_training_history.csv")
lowest_loss_epoch = training_history.loc[training_history["VALIDATION LOSS"].idxmin()]
highest_f1_epoch = training_history.loc[training_history["VALIDATION F1"].idxmax()]

selection_comparison = pd.DataFrame([
    {"SELECTION": "Lowest validation loss", **lowest_loss_epoch.to_dict()},
    {"SELECTION": "Highest validation F1", **highest_f1_epoch.to_dict()},
])
display(selection_comparison.round(4))
selection_comparison.to_csv(TABLES_DIR / "lstm_epoch_selection_comparison.csv", index=False)

交叉熵关注预测概率及其置信程度；F1 根据分类结果衡量正类的 Precision 和 Recall。两者最好的轮次可能不同。为保持此前的模型选择规则，这里继续以最低验证损失对应的模型进行分析；最高验证 F1 仅作为对照观察。

In [ ]:
# 加载按验证损失选择的模型并生成预测
import torch
from torch.utils.data import DataLoader
from sentiment_analysis.lstm import LSTMClassifier
from sentiment_analysis.paths import CHECKPOINTS_DIR, CLEANED_DIR
from sentiment_analysis.sequence_data import SentimentDataset, TOKEN_PATTERN, tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(CHECKPOINTS_DIR / "lstm_best.pt", map_location=device, weights_only=True)
if checkpoint["token_pattern"] != TOKEN_PATTERN.pattern or not checkpoint["lowercase"]:
    raise ValueError("Checkpoint tokenizer does not match the current sequence processing.")
if checkpoint["epoch"] != int(lowest_loss_epoch["EPOCH"]):
    raise ValueError("Checkpoint epoch does not match the training history.")

validation_data = pd.read_csv(CLEANED_DIR / "val.csv")
validation_dataset = SentimentDataset(validation_data, checkpoint["vocabulary"], checkpoint["max_length"])
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)

lstm_model = LSTMClassifier(**checkpoint["model_config"]).to(device)
lstm_model.load_state_dict(checkpoint["model_state_dict"])
lstm_model.eval()

y_prediction = []
with torch.no_grad():
    for token_ids, lengths, labels in validation_loader:
        logits = lstm_model(token_ids.to(device), lengths)
        y_prediction.extend(logits.argmax(dim=1).cpu().tolist())

print(f"Checkpoint epoch: {checkpoint['epoch']} | Predictions: {len(y_prediction):,}")

In [ ]:
# 按误分类类型统计原始评论长度和截断情况
error_analysis = validation_data[["text", "label"]].copy()
error_analysis["prediction"] = y_prediction
error_analysis["token_count"] = error_analysis["text"].apply(lambda text: len(tokenize(text)))
error_analysis["truncated"] = error_analysis["token_count"] > checkpoint["max_length"]

false_positive = (error_analysis["label"] == 0) & (error_analysis["prediction"] == 1)
false_negative = (error_analysis["label"] == 1) & (error_analysis["prediction"] == 0)
correct_prediction = error_analysis["label"] == error_analysis["prediction"]

rows = []
for name, mask in [("Correct", correct_prediction), ("False positive", false_positive),
                   ("False negative", false_negative)]:
    selected = error_analysis[mask]
    rows.append({
        "CASE": name,
        "COUNT": len(selected),
        "MEDIAN TOKENS": selected["token_count"].median(),
        "TRUNCATED COUNT": selected["truncated"].sum(),
        "TRUNCATED (%)": selected["truncated"].mean() * 100,
    })

error_summary = pd.DataFrame(rows)
display(error_summary.round(2))
error_summary.to_csv(TABLES_DIR / "lstm_error_length_summary.csv", index=False)

In [ ]:
# 分别查看误判为正面和误判为负面的评论
for name, mask in [("False positive", false_positive), ("False negative", false_negative)]:
    print(name)
    for index, row in error_analysis[mask].head(2).iterrows():
        print(f"Index: {index} | Label: {row['label']} | Prediction: {row['prediction']} | Tokens: {row['token_count']}")
        print(row["text"])
        print()

## 初步观察

- 第 3 轮的验证损失最低（0.3472），第 5 轮的验证 F1 最高（0.8744）；两种指标所选择的轮次不同。当前保留此前确定的最低验证损失规则。
- 按第 3 轮模型的预测结果，误判为正面的评论有 132 条，误判为负面的评论有 232 条。
- 误判为负面的评论中有 33 条超过 512 个词项，占该组 14.22%；预测正确的评论中该比例为 6.91%。这是关联性观察，不能单独证明截断导致误判。
- 查看少量样本发现，有的评论同时表达赞赏和批评；这只是对具体文本的描述，不能据此确定模型错误原因或概括全部样本。